In [3]:
import pandas as pd, numpy as np
import warnings
import os
import requests
import time
warnings.simplefilter(action='ignore', category=FutureWarning)
pd.set_option('display.max_columns', None)

#Change this to be wherever you want the data to go
os.chdir("/Users/owenstonge/Desktop/Data")

In [4]:
import re
import pandas as pd
from curl_cffi import requests as cffi_requests


def _clean_html_cols(df):
    
    if "Name" in df.columns:
        df["fg_playerid"] = df["Name"].astype(str).str.extract(r"playerid=(\d+)")
    if "Team" in df.columns:
        df["fg_teamid"] = df["Team"].astype(str).str.extract(r"team=(\d+)")

    for col in df.columns:
        if df[col].dtype == "object" and df[col].astype(str).str.contains("<a", na=False).any():
            df[col] = df[col].astype(str).str.replace(r"<[^>]+>", "", regex=True)
    return df


def fg_season_stats(start_season, end_season=None, stats="bat", qual="0",
                    ind=1, stat_type=8, league="all"):
    if end_season is None:
        end_season = start_season

    url = "https://www.fangraphs.com/api/leaders/major-league/data"
    params = {
        "age": "", "pos": "all", "stats": stats, "lg": league,
        "qual": qual, "season": end_season, "season1": start_season,
        "startdate": "", "enddate": "", "month": "0", "hand": "",
        "team": "0", "pageitems": "2000000000", "pagenum": "1",
        "ind": ind, "rost": "0", "players": "", "type": stat_type,
        "postseason": "", "sortdir": "default", "sortstat": "WAR",
    }

    r = cffi_requests.get(url, params=params, impersonate="chrome", timeout=30)
    r.raise_for_status()
    df = pd.DataFrame(r.json()["data"])
    return _clean_html_cols(df)


In [5]:
seasons = [2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026]

frames = {}
for yr in seasons:
    frames[yr] = fg_season_stats(yr, stats="pit", qual="0")
    time.sleep(2)

pit_all = pd.concat(frames.values(), ignore_index=True)

In [6]:
print(list(pit_all.columns))

['Throws', 'xMLBAMID', 'Name', 'Team', 'Season', 'Age', 'AgeR', 'SeasonMin', 'SeasonMax', 'W', 'L', 'ERA', 'G', 'GS', 'QS', 'CG', 'ShO', 'SV', 'BS', 'IP', 'TBF', 'H', 'R', 'ER', 'HR', 'BB', 'IBB', 'HBP', 'WP', 'BK', 'SO', 'GB', 'FB', 'LD', 'IFFB', 'Pitches', 'Balls', 'Strikes', 'RS', 'IFH', 'BU', 'BUH', 'K/9', 'BB/9', 'K/BB', 'H/9', 'HR/9', 'AVG', 'WHIP', 'BABIP', 'LOB%', 'FIP', 'GB/FB', 'LD%', 'GB%', 'FB%', 'IFFB%', 'HR/FB', 'IFH%', 'BUH%', 'TTO%', 'CFraming', 'Starting', 'Start-IP', 'Relieving', 'Relief-IP', 'RAR', 'WAR', 'Dollars', 'RA9-Wins', 'LOB-Wins', 'BIP-Wins', 'BS-Wins', 'tERA', 'xFIP', 'WPA', '-WPA', '+WPA', 'RE24', 'REW', 'pLI', 'inLI', 'gmLI', 'exLI', 'Pulls', 'Games', 'WPA/LI', 'Clutch', 'FB%1', 'FBv', 'SL%', 'SLv', 'CT%', 'CTv', 'CB%', 'CBv', 'CH%', 'CHv', 'SF%', 'SFv', 'KN%', 'KNv', 'XX%', 'PO%', 'wFB', 'wSL', 'wCT', 'wCB', 'wCH', 'wSF', 'wKN', 'wFB/C', 'wSL/C', 'wCT/C', 'wCB/C', 'wCH/C', 'wSF/C', 'wKN/C', 'O-Swing%', 'Z-Swing%', 'Swing%', 'O-Contact%', 'Z-Contact%', 'C

In [7]:
pit_all

,Throws,xMLBAMID,Name,Team,Season,Age,AgeR,SeasonMin,SeasonMax,W,L,ERA,G,GS,QS,CG,ShO,SV,BS,IP,TBF,H,R,ER,HR,BB,IBB,HBP,WP,BK,SO,GB,FB,LD,IFFB,Pitches,Balls,Strikes,RS,IFH,BU,BUH,K/9,BB/9,K/BB,H/9,HR/9,AVG,WHIP,BABIP,LOB%,FIP,GB/FB,LD%,GB%,FB%,IFFB%,HR/FB,IFH%,BUH%,TTO%,CFraming,Starting,Start-IP,Relieving,Relief-IP,RAR,WAR,Dollars,RA9-Wins,LOB-Wins,BIP-Wins,BS-Wins,tERA,xFIP,WPA,-WPA,+WPA,RE24,REW,pLI,inLI,gmLI,exLI,Pulls,Games,WPA/LI,Clutch,FB%1,FBv,SL%,SLv,CT%,CTv,CB%,CBv,CH%,CHv,SF%,SFv,KN%,KNv,XX%,PO%,wFB,wSL,wCT,wCB,wCH,wSF,wKN,wFB/C,wSL/C,wCT/C,wCB/C,wCH/C,wSF/C,wKN/C,O-Swing%,Z-Swing%,Swing%,O-Contact%,Z-Contact%,Contact%,Zone%,F-Strike%,SwStr%,CStr%,C+SwStr%,HLD,SD,MD,ERA-,FIP-,xFIP-,K%,BB%,K-BB%,SIERA,kwERA,RS/9,E-F,Pull,Cent,Oppo,Soft,Med,Hard,bipCount,Pull%,Cent%,Oppo%,Soft%,Med%,Hard%,K/9+,BB/9+,K/BB+,H/9+,HR/9+,AVG+,WHIP+,BABIP+,LOB%+,K%+,BB%+,LD%+,GB%+,FB%+,HRFB%+,Pull%+,Cent%+,Oppo%+,Soft%+,Med%+,Hard%+,xERA,pb_o_CH,pb_s_CH,pb_c_CH,pb_o_CU,pb_s_CU,pb_c_CU,pb_o_FF,pb_s_FF,pb_c_FF,pb_o_SI,pb_s_SI,pb_c_SI,pb_o_SL,pb_s_SL,pb_c_SL,pb_o_KC,pb_s_KC,pb_c_KC,pb_o_FC,pb_s_FC,pb_c_FC,pb_o_FS,pb_s_FS,pb_c_FS,pb_overall,pb_stuff,pb_command,pb_xRV100,pb_ERA,sp_s_CH,sp_l_CH,sp_p_CH,sp_s_CU,sp_l_CU,sp_p_CU,sp_s_FF,sp_l_FF,sp_p_FF,sp_s_SI,sp_l_SI,sp_p_SI,sp_s_SL,sp_l_SL,sp_p_SL,sp_s_KC,sp_l_KC,sp_p_KC,sp_s_FC,sp_l_FC,sp_p_FC,sp_s_FS,sp_l_FS,sp_p_FS,sp_s_FO,sp_l_FO,sp_p_FO,sp_stuff,sp_location,sp_pitching,PPTV,CPTV,BPTV,DSV,DGV,BTV,rPPTV,rCPTV,rBPTV,rDSV,rDGV,rBTV,EBV,ESV,rFTeamV,rBTeamV,rTV,BReview,BOverturned,BCorrect%,wBReview,PReview,POverturned,PCorrect%,wPReview,CReview,COverturned,CCorrect%,wCReview,pfxFA%,pfxFT%,pfxFC%,pfxFS%,pfxFO%,pfxSI%,pfxSL%,pfxCU%,pfxKC%,pfxEP%,pfxCH%,pfxSC%,pfxKN%,pfxUN%,pfxSLO%,pfxST%,pfxCUO%,pfxCV%,pfxvFA,pfxvFT,pfxvFC,pfxvFS,pfxvFO,pfxvSI,pfxvSL,pfxvCU,pfxvKC,pfxvEP,pfxvCH,pfxvSC,pfxvKN,pfxvSLO,pfxvST,pfxvCUO,pfxvCV,pfxFA-X,pfxFT-X,pfxFC-X,pfxFS-X,pfxFO-X,pfxSI-X,pfxSL-X,pfxCU-X,pfxKC-X,pfxEP-X,pfxCH-X,pfxSC-X,pfxKN-X,pfxSLO-X,pfxST-X,pfxCUO-X,pfxCV-X,pfxFA-Z,pfxFT-Z,pfxFC-Z,pfxFS-Z,pfxFO-Z,pfxSI-Z,pfxSL-Z,pfxCU-Z,pfxKC-Z,pfxEP-Z,pfxCH-Z,pfxSC-Z,pfxKN-Z,pfxSLO-Z,pfxST-Z,pfxCUO-Z,pfxCV-Z,pfxwFA,pfxwFT,pfxwFC,pfxwFS,pfxwFO,pfxwSI,pfxwSL,pfxwCU,pfxwKC,pfxwEP,pfxwCH,pfxwSC,pfxwKN,pfxwSLO,pfxwST,pfxwCUO,pfxwCV,pfxwFA/C,pfxwFT/C,pfxwFC/C,pfxwFS/C,pfxwFO/C,pfxwSI/C,pfxwSL/C,pfxwCU/C,pfxwKC/C,pfxwEP/C,pfxwCH/C,pfxwSC/C,pfxwKN/C,pfxwSLO/C,pfxwST/C,pfxwCUO/C,pfxwCV/C,pfxaaFA,pfxaaFT,pfxaaFC,pfxaaFS,pfxaaFO,pfxaaSI,pfxaaSL,pfxaaCU,pfxaaKC,pfxaaEP,pfxaaCH,pfxaaSC,pfxaaKN,pfxaaSLO,pfxaaST,pfxaaCUO,pfxaaCV,pfxspFA,pfxspFT,pfxspFC,pfxspFS,pfxspFO,pfxspSI,pfxspSL,pfxspCU,pfxspKC,pfxspEP,pfxspCH,pfxspSC,pfxspKN,pfxspSLO,pfxspST,pfxspCUO,pfxspCV,pfxO-Swing%,pfxZ-Swing%,pfxSwing%,pfxO-Contact%,pfxZ-Contact%,pfxContact%,pfxZone%,pfxPace,AvgBatSpeed,FastSwing%,SwingLength,SquaredUpContact%,SquaredUpSwing%,BlastContact%,BlastSwing%,Swords,CompetitiveSwings,Tilt,AttackAngle,AttackDirection,IdealAttackAngle%,DepthInBox,DistanceOffPlate,scH-Swing%,scH-Contact%,scH-Zone%,scS-Swing%,scS-Contact%,scS-Zone%,scC-Swing%,scC-Contact%,scC-Zone%,scW-Swing%,scW-Contact%,scW-Zone%,scSI-Swing%,scSI-Contact%,scSI-Zone%,scSO-Swing%,scSO-Contact%,scSO-Zone%,scO-Swing%,scO-Contact%,scO-Zone%,scZ-Swing%,scZ-Contact%,scZ-Zone%,piCH%,piCS%,piCU%,piFA%,piFC%,piFS%,piKN%,piSB%,piSI%,piSL%,piXX%,pivCH,pivCS,pivCU,pivFA,pivFC,pivFS,pivKN,pivSB,pivSI,pivSL,pivXX,piCH-X,piCS-X,piCU-X,piFA-X,piFC-X,piFS-X,piKN-X,piSB-X,piSI-X,piSL-X,piXX-X,piCH-Z,piCS-Z,piCU-Z,piFA-Z,piFC-Z,piFS-Z,piKN-Z,piSB-Z,piSI-Z,piSL-Z,piXX-Z,piwCH,piwCS,piwCU,piwFA,piwFC,piwFS,piwKN,piwSB,piwSI,piwSL,piwXX,piwCH/C,piwCS/C,piwCU/C,piwFA/C,piwFC/C,piwFS/C,piwKN/C,piwSB/C,piwSI/C,piwSL/C,piwXX/C,piO-Swing%,piZ-Swing%,piSwing%,piO-Contact%,piZ-Contact%,piContact%,piZone%,piPace,Events,EV,LA,Barrels,Barrel%,maxEV,HardHit,HardHit%,Q,TG,TIP,PlayerNameRoute,PlayerName,positionDB,position,TeamName,TeamNameAbb,teamid,playerTeamId,playerid,EV90,fg_playerid,fg_teamid
0,L,477132,Clay

In [29]:
#Save as parquet (more memory efficient than a CSV)
pit_all.to_parquet("FinalProject.parquet", index=False)